<!-- # 🚀 EMP 자동화 시스템
달랏마트 ERP 시스템 자동화를 위한 완전한 코드

## 📋 단계별 실행 가이드:
1. **라이브러리 설치** (셀 1)
2. **필요 모듈 import** (셀 2)  
3. **EMP 프로세스 연결** (셀 3)
4. **진짜 EMP 시스템 찾기** (셀 4)
5. **버튼 조작 및 자동화** (셀 5~) -->


In [1]:
# 🔧 1단계: 필요한 라이브러리 설치
!pip install pywinauto pyautogui psutil



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\jusun\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# 📚 2단계: 필요 모듈 import
import os
import time
import sys
import traceback
import psutil
from pywinauto import Application, findwindows
import pyautogui

print("✅ 모든 모듈이 성공적으로 로드되었습니다!")


✅ 모든 모듈이 성공적으로 로드되었습니다!


In [3]:
# 🔌 3단계: EMP 프로세스 연결

def find_emp_process():
    """실행 중인 EMP 프로세스를 찾아 PID 반환"""
    for p in psutil.process_iter(['pid', 'name', 'exe']):
        try:
            name = (p.info['name'] or '').lower()
            if 'emp' in name and '.exe' in name:
                print(f"✅ EMP 프로세스 발견: {p.info['name']}, PID: {p.info['pid']}")
                return p.info['pid']
        except:
            continue
    return None

def connect_to_emp():
    """EMP 애플리케이션에 연결"""
    pid = find_emp_process()
    
    if pid:
        app = Application(backend="uia").connect(process=pid)
        print(f"✅ EMP 프로세스(PID: {pid})에 연결 성공!")
        return app
    else:
        print("❌ EMP 프로세스를 찾을 수 없습니다. EMP를 먼저 실행해주세요.")
        return None

# EMP에 연결
app = connect_to_emp()


✅ EMP 프로세스 발견: EMP.Startup.exe, PID: 3100
✅ EMP 프로세스(PID: 3100)에 연결 성공!


In [4]:
# 🎯 4단계: 진짜 EMP 시스템 창 찾기

def find_emp_window(app):
    """진짜 EMP 시스템 창을 찾기"""
    print("🔍 EMP 관련 창들을 검색 중...")
    
    all_windows = findwindows.find_windows(title_re=r".*")
    
    for handle in all_windows:
        try:
            window = app.window(handle=handle)
            title = window.window_text()
            class_name = window.class_name()
            
            # 달랏마트 EMP 시스템 찾기
            if '달랏마트' in title and 'Enhanced Management Plus' in title:
                print(f"🎯 진짜 EMP 시스템 발견!")
                print(f"   제목: {title}")
                print(f"   클래스: {class_name}")
                print(f"   핸들: {handle}")
                return window
                
        except:
            continue
    
    print("❌ EMP 시스템 창을 찾을 수 없습니다.")
    return None

# 진짜 EMP 창 찾기
if app:
    emp_window = find_emp_window(app)
    if emp_window:
        print("\n✅ EMP 시스템 연결 완료!")
    else:
        print("\n❌ EMP 시스템을 찾지 못했습니다.")
else:
    print("❌ 먼저 EMP 프로세스에 연결해주세요.")


🔍 EMP 관련 창들을 검색 중...
🎯 진짜 EMP 시스템 발견!
   제목: 달랏마트 (신규) - 달랏마트 : Enhanced Management Plus +
   클래스: WindowsForms10.Window.8.app.0.70a1e_r7_ad1
   핸들: 203696

✅ EMP 시스템 연결 완료!


In [9]:
# 🔍 5단계: 버튼 탐색 및 조작 함수들

def find_all_buttons(win, show_all=True):
    """창의 모든 버튼을 찾아서 상세 정보 출력"""
    print("🔍 EMP 버튼들을 탐색 중...")
    buttons = win.descendants(control_type="Button")
    print(f"총 {len(buttons)}개의 버튼을 발견했습니다!")
    print("-" * 60)
    
    available_buttons = []
    all_buttons_info = []
    
    for i, btn in enumerate(buttons):
        try:
            text = btn.window_text()
            is_enabled = btn.is_enabled()
            is_visible = btn.is_visible()
            
            # 모든 버튼 정보 저장
            if text:
                all_buttons_info.append((i, text, is_enabled, is_visible))
            
            if text and is_enabled and is_visible:
                available_buttons.append((i, text))
                if show_all:
                    print(f"✅ [{i:2d}] '{text}'")
            elif text:
                if show_all:
                    status = "❌비활성" if not is_enabled else "👁️숨김"
                    print(f"{status} [{i:2d}] '{text}'")
                
        except:
            if show_all:
                print(f"⚠️ [{i:2d}] [버튼 정보 읽기 실패]")
    
    print("-" * 60)
    print(f"🎯 사용 가능한 버튼: {len(available_buttons)}개")
    print(f"📋 텍스트가 있는 버튼: {len(all_buttons_info)}개")
    print(f"📊 전체 버튼: {len(buttons)}개")
    
    return buttons, available_buttons, all_buttons_info

def click_button_by_index(buttons, index):
    """버튼 번호로 클릭"""
    try:
        if 0 <= index < len(buttons):
            button = buttons[index]
            text = button.window_text()
            
            if button.is_enabled() and button.is_visible():
                button.click()
                print(f"✅ 버튼 클릭 성공: [{index}] '{text}'")
                time.sleep(0.5)  # 클릭 후 잠시 대기
                return True
            else:
                print(f"❌ 버튼이 비활성화되어 있습니다: [{index}] '{text}'")
                return False
        else:
            print(f"❌ 잘못된 버튼 번호: {index}")
            return False
    except Exception as e:
        print(f"❌ 버튼 클릭 실패: {e}")
        return False

def click_button_by_text(win, button_text):
    """버튼 텍스트로 클릭"""
    try:
        button = win.child_window(title=button_text, control_type="Button")
        if button.is_enabled() and button.is_visible():
            button.click()
            print(f"✅ 버튼 클릭 성공: '{button_text}'")
            time.sleep(0.5)
            return True
        else:
            print(f"❌ 버튼이 비활성화되어 있습니다: '{button_text}'")
            return False
    except Exception as e:
        print(f"❌ 버튼 클릭 실패: {e}")
        return False

print("✅ 버튼 조작 함수들이 준비되었습니다!")


✅ 버튼 조작 함수들이 준비되었습니다!


In [14]:
# 🔍 8단계: 모든 컨트롤 완전 탐색

def find_all_controls_complete(win):
    """모든 종류의 컨트롤을 찾아서 완전히 분석"""
    print("🔍 EMP의 모든 컨트롤을 탐색 중...")
    print("=" * 80)
    
    # 모든 컨트롤 타입들
    control_types = [
        "Button", "MenuItem", "Text", "Hyperlink", "ListItem", 
        "TreeItem", "TabItem", "Static", "Group", "Pane", 
        "Window", "MenuBar", "ToolBar", "Edit", "ComboBox",
        "ListBox", "CheckBox", "RadioButton", "Slider", "Document"
    ]
    
    all_controls = []
    controls_by_type = {}
    
    for control_type in control_types:
        try:
            controls = win.descendants(control_type=control_type)
            controls_count = len(controls)
            controls_by_type[control_type] = controls_count
            
            print(f"📋 {control_type:<15}: {controls_count:3d}개")
            
            # 각 컨트롤의 텍스트 정보 수집
            for i, ctrl in enumerate(controls):
                try:
                    # 다양한 텍스트 속성 시도
                    text_info = {}
                    
                    # window_text 시도
                    try:
                        window_text = ctrl.window_text()
                        if window_text and window_text.strip():
                            text_info['window_text'] = window_text.strip()
                    except: pass
                    
                    # automation_id 시도
                    try:
                        auto_id = ctrl.automation_id()
                        if auto_id and auto_id.strip():
                            text_info['automation_id'] = auto_id.strip()
                    except: pass
                    
                    # class_name 시도
                    try:
                        class_name = ctrl.class_name()
                        if class_name and class_name.strip():
                            text_info['class_name'] = class_name.strip()
                    except: pass
                    
                    # element_info.name 시도
                    try:
                        element_name = ctrl.element_info.name
                        if element_name and element_name.strip():
                            text_info['element_name'] = element_name.strip()
                    except: pass
                    
                    if text_info:
                        all_controls.append({
                            'type': control_type,
                            'index': i,
                            'element': ctrl,
                            'texts': text_info
                        })
                        
                except Exception as e:
                    continue
                    
        except Exception as e:
            print(f"❌ {control_type:<15}: 검색 실패 - {e}")
    
    print("=" * 80)
    print(f"📊 총 발견된 컨트롤: {len(all_controls)}개")
    
    return all_controls, controls_by_type

def search_controls(all_controls, keywords=None, control_type_filter=None):
    """컨트롤에서 특정 키워드나 타입으로 검색"""
    if keywords is None:
        keywords = ['상품', '관리', '단일옵션', '품목', '재고', '등록', '마스터', '기초정보', '입고', '출고', '판매', '구매']
    
    matches = []
    
    for ctrl_info in all_controls:
        # 타입 필터 적용
        if control_type_filter and ctrl_info['type'] != control_type_filter:
            continue
            
        # 키워드 검색
        for text_type, text_value in ctrl_info['texts'].items():
            for keyword in keywords:
                if keyword in str(text_value):
                    matches.append({
                        'type': ctrl_info['type'],
                        'index': ctrl_info['index'],
                        'text_type': text_type,
                        'text_value': text_value,
                        'element': ctrl_info['element']
                    })
                    break
    
    return matches

def display_all_controls(all_controls, limit=50):
    """모든 컨트롤의 텍스트 정보 표시"""
    print(f"\n📋 모든 컨트롤 텍스트 정보 (처음 {limit}개):")
    print("-" * 100)
    
    displayed = 0
    for i, ctrl_info in enumerate(all_controls):
        if displayed >= limit:
            break
            
        print(f"[{i:3d}] {ctrl_info['type']:<12}")
        for text_type, text_value in ctrl_info['texts'].items():
            if len(text_value) > 0:
                print(f"      {text_type:<15}: '{text_value}'")
                displayed += 1
                if displayed >= limit:
                    break
        print()

def click_control(ctrl_element, method='click'):
    """컨트롤 클릭 (다양한 방법 시도)"""
    try:
        if method == 'click':
            ctrl_element.click()
        elif method == 'double_click':
            ctrl_element.double_click()
        elif method == 'right_click':
            ctrl_element.right_click()
        
        print(f"✅ 컨트롤 {method} 성공!")
        time.sleep(0.5)
        return True
    except Exception as e:
        print(f"❌ 컨트롤 {method} 실패: {e}")
        return False

# EMP 창에서 모든 컨트롤 탐색 실행
if 'emp_window' in locals() and emp_window:
    all_controls, controls_by_type = find_all_controls_complete(emp_window)
    
    # 키워드로 검색
    matches = search_controls(all_controls)
    
    if matches:
        print(f"\n🎯 키워드 매칭 결과: {len(matches)}개")
        for i, match in enumerate(matches):
            print(f"[{i:2d}] {match['type']:<12} {match['text_type']:<15}: '{match['text_value']}'")
    else:
        print("\n❌ 키워드 매칭 결과가 없습니다.")
    
    # 모든 컨트롤 표시
    display_all_controls(all_controls, limit=30)
    
    print(f"\n💡 사용법:")
    print(f"   search_controls(all_controls, ['원하는키워드'])  # 키워드로 검색")
    print(f"   click_control(matches[0]['element'])           # 첫 번째 매칭 요소 클릭")
    
else:
    print("❌ 먼저 EMP 창에 연결해주세요.")


🔍 EMP의 모든 컨트롤을 탐색 중...
📋 Button         :  48개
📋 MenuItem       :   2개
📋 Text           :  16개
📋 Hyperlink      :   0개
📋 ListItem       :   0개
📋 TreeItem       :   0개
📋 TabItem        :   3개
❌ Static         : 검색 실패 - 'Static'
📋 Group          :   0개
📋 Pane           :  38개
📋 Window         :   1개
📋 MenuBar        :   1개
📋 ToolBar        :   6개
📋 Edit           :  41개
📋 ComboBox       :   1개
❌ ListBox        : 검색 실패 - 'ListBox'
📋 CheckBox       :   2개
📋 RadioButton    :   0개
📋 Slider         :   0개
📋 Document       :   0개
📊 총 발견된 컨트롤: 144개

🎯 키워드 매칭 결과: 12개
[ 0] Button       window_text    : '상품그룹'
[ 1] Button       element_name   : '상품그룹'
[ 2] Text         window_text    : '※ 형식 : [상품코드] [대표코드] [상품명] [사이즈체계코드] [브랜드코드] [년도코드] [시즌코드] [성별코드] [아이템코드] [택가] [정상가] [판매가] [원가] (상제품) (입고업체) (상품참고사항1) (상품참고사항2) (상품참고사항3) (상품참고사항4) (상품참고사항5) (상품참고사항6) (상품참고사항7) (상품참고사항8) (상품참고사항9) (상품참고사항10) (상품참고사항11) (상품참고사항12) (상품참고사항13) (상품참고사항14) (상품참고사항15)'
[ 3] Text         element_name   : '※ 형식 : [상품코드] 

In [18]:
search_controls(all_controls,['발주'])

[]

In [19]:
# 🔍 "발주 등록" 버튼 찾기

def find_specific_text(all_controls, search_text):
    """특정 텍스트를 포함한 컨트롤 찾기"""
    print(f"🔍 '{search_text}' 텍스트를 찾는 중...")
    matches = []
    
    for ctrl_info in all_controls:
        for text_type, text_value in ctrl_info['texts'].items():
            if search_text in str(text_value):
                matches.append({
                    'type': ctrl_info['type'],
                    'index': ctrl_info['index'],
                    'text_type': text_type,
                    'text_value': text_value,
                    'element': ctrl_info['element']
                })
                print(f"🎯 발견: [{ctrl_info['type']}] {text_type}: '{text_value}'")
    
    return matches

def find_menu_by_partial_text(all_controls, partial_texts):
    """부분 텍스트로 메뉴 찾기"""
    print(f"🔍 부분 텍스트로 메뉴 검색: {partial_texts}")
    matches = []
    
    for ctrl_info in all_controls:
        for text_type, text_value in ctrl_info['texts'].items():
            text_lower = str(text_value).lower()
            
            # 모든 부분 텍스트가 포함되어 있는지 확인
            if all(partial.lower() in text_lower for partial in partial_texts):
                matches.append({
                    'type': ctrl_info['type'],
                    'index': ctrl_info['index'],
                    'text_type': text_type,
                    'text_value': text_value,
                    'element': ctrl_info['element']
                })
                print(f"🎯 매칭: [{ctrl_info['type']}] '{text_value}'")
    
    return matches

# 발주 등록 관련 검색 실행
if 'all_controls' in locals():
    print("=" * 60)
    
    # 1. "발주 등록" 정확한 텍스트로 검색
    exact_matches = find_specific_text(all_controls, "발주 등록")
    
    # 2. "발주"와 "등록" 부분 텍스트로 검색
    partial_matches = find_menu_by_partial_text(all_controls, ["발주", "등록"])
    
    # 3. "발주" 관련 모든 항목 검색
    order_matches = find_specific_text(all_controls, "발주")
    
    print("=" * 60)
    print("📊 검색 결과 요약:")
    print(f"   정확한 매칭 ('발주 등록'): {len(exact_matches)}개")
    print(f"   부분 매칭 ('발주' + '등록'): {len(partial_matches)}개") 
    print(f"   발주 관련 전체: {len(order_matches)}개")
    
    # 결과가 있으면 클릭 방법 안내
    if exact_matches:
        print("\n🎉 '발주 등록' 버튼을 찾았습니다!")
        print("클릭하려면:")
        print(f"   click_control(exact_matches[0]['element'])")
    elif partial_matches:
        print("\n🎯 발주+등록 관련 항목을 찾았습니다!")
        print("클릭하려면:")
        print(f"   click_control(partial_matches[0]['element'])")
    elif order_matches:
        print("\n📋 발주 관련 항목들:")
        for i, match in enumerate(order_matches):
            print(f"   [{i}] {match['type']}: '{match['text_value']}'")
        print("\n클릭하려면:")
        print(f"   click_control(order_matches[번호]['element'])")
    else:
        print("\n❌ 발주 등록 관련 항목을 찾지 못했습니다.")
        print("💡 화면에서 발주 메뉴를 수동으로 열어보세요.")
        
else:
    print("❌ 먼저 8번 셀을 실행해서 all_controls를 생성해주세요.")


🔍 '발주 등록' 텍스트를 찾는 중...
🔍 부분 텍스트로 메뉴 검색: ['발주', '등록']
🔍 '발주' 텍스트를 찾는 중...
📊 검색 결과 요약:
   정확한 매칭 ('발주 등록'): 0개
   부분 매칭 ('발주' + '등록'): 0개
   발주 관련 전체: 0개

❌ 발주 등록 관련 항목을 찾지 못했습니다.
💡 화면에서 발주 메뉴를 수동으로 열어보세요.


In [20]:
# 🔍 EMP 메뉴 광범위 검색 (발주 등록 찾기)

def search_all_menus(all_controls):
    """모든 가능한 메뉴 관련 키워드로 검색"""
    
    # 발주 관련 다양한 키워드들
    keywords = [
        '발주', '주문', '구매', '등록', '입력', '신규',
        '오더', 'order', 'purchase', '매입', '조달',
        '요청', '신청', '의뢰', '접수', '수주'
    ]
    
    # 메뉴 관련 키워드들  
    menu_keywords = [
        '관리', '등록', '입력', '조회', '수정', '삭제',
        '마스터', '기초', '정보', '데이터', '처리'
    ]
    
    print("🔍 광범위 키워드 검색 중...")
    print("-" * 70)
    
    all_matches = {}
    
    for keyword in keywords + menu_keywords:
        matches = []
        for ctrl_info in all_controls:
            for text_type, text_value in ctrl_info['texts'].items():
                if keyword in str(text_value):
                    matches.append({
                        'type': ctrl_info['type'],
                        'text_value': text_value,
                        'element': ctrl_info['element']
                    })
        
        if matches:
            all_matches[keyword] = matches
            print(f"📌 '{keyword}': {len(matches)}개 발견")
            for match in matches[:3]:  # 처음 3개만 표시
                print(f"   └ [{match['type']}] '{match['text_value']}'")
    
    return all_matches

def show_tree_structure(win):
    """왼쪽 트리 메뉴 구조 상세 분석"""
    print("\n🌲 트리 메뉴 구조 분석:")
    print("-" * 70)
    
    try:
        # TreeView 컨트롤 찾기
        tree_views = win.descendants(control_type="Tree")
        print(f"TreeView 발견: {len(tree_views)}개")
        
        for tv_idx, tree_view in enumerate(tree_views):
            print(f"\n🌳 TreeView #{tv_idx+1}:")
            
            # 트리 아이템들 분석
            tree_items = tree_view.descendants(control_type="TreeItem")
            print(f"   TreeItem 개수: {len(tree_items)}개")
            
            for i, item in enumerate(tree_items):
                try:
                    text = item.window_text()
                    if text and text.strip():
                        # 발주/구매/주문 관련 키워드 체크
                        purchase_keywords = ['발주', '구매', '주문', '매입', '조달', '입고']
                        if any(keyword in text for keyword in purchase_keywords):
                            print(f"   🎯 [{i:2d}] '{text}' ⭐")
                        else:
                            print(f"   📄 [{i:2d}] '{text}'")
                except:
                    print(f"   ❌ [{i:2d}] [텍스트 읽기 실패]")
                    
    except Exception as e:
        print(f"❌ 트리 구조 분석 실패: {e}")

# 실행
if 'all_controls' in locals():
    # 1. 광범위 키워드 검색
    all_matches = search_all_menus(all_controls)
    
    # 2. 트리 메뉴 구조 분석
    if 'emp_window' in locals() and emp_window:
        show_tree_structure(emp_window)
    
    print("\n💡 발주 등록을 찾는 방법:")
    print("1. 위 결과에서 '구매', '주문', '매입' 관련 항목들을 확인")
    print("2. 트리 메뉴에서 ⭐ 표시된 항목들을 클릭해보기")
    print("3. 해당 항목을 클릭한 후 하위 메뉴에서 '등록' 찾기")
    
else:
    print("❌ 먼저 8번 셀을 실행해주세요.")


🔍 광범위 키워드 검색 중...
----------------------------------------------------------------------
📌 '등록': 2개 발견
   └ [Text] '일괄등록'
   └ [Text] '일괄등록'
📌 '신규': 4개 발견
   └ [Button] '신규(F3)'
   └ [Button] '신규(F3)'
   └ [Button] '신규(F3)'
📌 '관리': 2개 발견
   └ [Window] '상품 관리'
   └ [Window] '상품 관리'
📌 '등록': 2개 발견
   └ [Text] '일괄등록'
   └ [Text] '일괄등록'
📌 '조회': 8개 발견
   └ [Button] '조회(F2)'
   └ [Button] '조회(F2)'
   └ [Button] '조회(F2)'
📌 '수정': 18개 발견
   └ [Button] '수정(F4)'
   └ [Button] '수정(F4)'
   └ [Button] '수정(F4)'
📌 '삭제': 4개 발견
   └ [Button] '삭제(F5)'
   └ [Button] '삭제(F5)'
   └ [Button] '삭제(F5)'

🌲 트리 메뉴 구조 분석:
----------------------------------------------------------------------
TreeView 발견: 0개

💡 발주 등록을 찾는 방법:
1. 위 결과에서 '구매', '주문', '매입' 관련 항목들을 확인
2. 트리 메뉴에서 ⭐ 표시된 항목들을 클릭해보기
3. 해당 항목을 클릭한 후 하위 메뉴에서 '등록' 찾기
